Język: [English](lab.en.ipynb) | **Polski**

# Laboratorium 15 — lokalna usługa inferencyjna z monitoringiem (EuroSAT + ResNet18)

Materiał towarzyszący [`README.pl.md`](README.pl.md). **Przed uruchomieniem tego
notebooka**: uzupełnij TODO w `app/model.py`, `app/server.py`,
`benchmark.py`, `monitoring.py` (patrz README sekcje 2-3, 5-6), i uruchom
`prepare_artifact.py`, jeśli jeszcze nie uruchomiony.

In [ ]:
import io  # noqa: F401 - uzywane w TODO ponizej
import json  # noqa: F401 - uzywane w TODO ponizej
import os
import sys
import time  # noqa: F401 - uzywane w TODO ponizej
from pathlib import Path

import matplotlib.pyplot as plt  # noqa: F401 - uzywane w TODO ponizej
import torch
import torchvision.datasets as tvd
from fastapi.testclient import TestClient
from torchvision.models import ResNet18_Weights

_REPO_ROOT = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "pyproject.toml").is_file()
    ),
    Path.cwd(),
)
DATA_CACHE_DIR = Path(
    os.environ.get("DEEP_LEARNING_DATA_CACHE", _REPO_ROOT / "data_cache")
).expanduser().resolve()

sys.path.insert(0, ".")

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Uzywane urzadzenie: {device}")

weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
full_ds = tvd.EuroSAT(root=DATA_CACHE_DIR, download=True, transform=preprocess)
CLASS_NAMES = full_ds.classes
print(f"EuroSAT: {len(full_ds)} obrazow, klasy: {CLASS_NAMES}")

## 1. Artefakt modelu — weryfikacja zgodności

`prepare_artifact.py` jest gotowy i powinien być już uruchomiony
(`uv run python labs/15_production/prepare_artifact.py`).

In [ ]:
# TODO:
# 1. from app import model as model_module
# 2. loaded = model_module.load_artifact("artifacts")
# 3. wybierz jeden obraz testowy (PIL, NIE z full_ds[i][0] ktory juz ma
#    preprocessing -- potrzebujesz surowego obrazu do zakodowania jako PNG)
# 4. zakoduj go do PNG bytes (io.BytesIO + img.save(..., format="PNG"))
# 5. policz prediction DWOMA sposobami: model_module.predict(loaded, png_bytes)
#    oraz "recznie" (preprocess + forward + softmax) -- porownaj, powinny
#    byc identyczne (to NIE jest jeszcze test przez API -- to tylko Twoj
#    kod z app/model.py wywolany dwa razy)
raise NotImplementedError("TODO: weryfikacja zgodnosci artefaktu")

## 2-3. `app/model.py` i `app/server.py`

Uzupełnione **poza tym notebookiem**, w plikach `app/model.py` i
`app/server.py` (README sekcje 2-3). Poniżej: zbudowanie klienta i
sprawdzenie, że endpointy odpowiadają.

In [ ]:
from app.server import app as fastapi_app

with TestClient(fastapi_app) as client:
    r_health = client.get("/health")
    r_ready = client.get("/ready")
    r_info = client.get("/model-info")
    print(f"GET /health: {r_health.status_code} {r_health.json()}")
    print(f"GET /ready: {r_ready.status_code} {r_ready.json()}")
    print(f"GET /model-info: {r_info.status_code} {r_info.json()}")

## 4. Zgodność API z bezpośrednim wywołaniem — Twoje zadanie

In [ ]:
# TODO:
# 1. wybierz prawdziwy obraz EuroSAT (PIL, surowy, bez preprocessingu),
#    zakoduj do PNG bytes
# 2. wyslij POST /predict przez `client` (uzyj "with TestClient(fastapi_app) as client:")
# 3. policz bezposrednio model_module.predict(loaded, png_bytes) na TYCH SAMYCH bytes
# 4. porownaj -- MUSZA sie zgadzac DOKLADNIE (ta sama klasa i confidence)
raise NotImplementedError("TODO: zgodnosc API vs bezposrednie wywolanie")

In [ ]:
# TODO: sprawdz obsluge bledow:
# 1. uszkodzony plik (np. b"not an image") -> jaki kod HTTP i komunikat?
# 2. plik za duzy (np. b"0" * 6_000_000) -> jaki kod HTTP?
# 3. upewnij sie, ze komunikat NIE zawiera pelnego tracebacka/sciezek lokalnych
raise NotImplementedError("TODO: obsluga bledow wejscia")

## 5. Benchmark: latency i throughput — Twoje zadanie

`send_requests` jest gotowe. `compute_stats` (w `benchmark.py`) jest Twoim
zadaniem (README sekcja 5).

In [ ]:
from benchmark import compute_stats, send_requests  # noqa: F401 - uzywane w TODO ponizej

# TODO:
# 1. zakoduj jeden prawdziwy obraz EuroSAT do PNG bytes (jak wyzej)
# 2. z `with TestClient(fastapi_app) as client:`, zmierz COLD (1. zadanie,
#    osobno) i WARM (20-30 kolejnych zadan, przez send_requests)
# 3. compute_stats(warm_times) -- wypisz mean/p50/p95/throughput
# 4. skomentuj: jak cold ma sie do warm? (patrz wyklad dla kontekstu)
raise NotImplementedError("TODO: benchmark cold/warm")

## 6. PSI — weryfikacja na znanym przykładzie

Zanim użyjesz `psi()` (Twoja implementacja w `monitoring.py`) na realnych
danych, zweryfikuj ją na znanym przykładzie z wykładu.

In [ ]:
from monitoring import histogram_counts, psi  # noqa: F401 - uzywane w TODO ponizej

# TODO:
# ref = torch.tensor([50.0, 50.0]); cur = torch.tensor([70.0, 30.0])
# oczekiwany wynik: 0.169460 (z lecture.md) -- zweryfikuj swoja psi()
# Sprawdz TEZ: psi(ref, ref) powinno dac 0 (identyczne rozklady)
# Sprawdz TEZ: pusty bin (np. cur=[100.0, 0.0]) -- z Twoim eps, czy wynik
# jest skonczony (nie inf/nan)?
raise NotImplementedError("TODO: weryfikacja PSI na znanym przykladzie")

## 7. Scenariusze monitoringu A/B/C — Twoje zadanie

Zobacz README sekcja 7 dla pełnych instrukcji. **WAŻNE**: wybierz próg PSI
i siłę transformacji PRZED sprawdzeniem accuracy w scenariuszu C, nie po.

In [ ]:
# Scenariusz A: kontrola (ta sama dystrybucja). Dane juz podzielone nizej.
torch.manual_seed(0)
perm = torch.randperm(len(full_ds))
test_idx = perm[2200:2600]
control_idx = perm[2600:3000]


def load_subset(indices):
    X = torch.stack([full_ds[i][0] for i in indices])
    y = torch.tensor([full_ds[i][1] for i in indices])
    return X, y


X_test, y_test = load_subset(test_idx.tolist())
X_control, y_control = load_subset(control_idx.tolist())

# TODO:
# 1. policz brightness_feature(X_test) jako referencja, zbuduj bin_edges
# 2. policz histogram_counts dla brightness_feature(X_control)
# 3. psi(ref_counts, control_counts) -- oczekiwany wynik: NISKI (brak alarmu)
raise NotImplementedError("TODO: scenariusz A -- kontrola")

In [ ]:
# Scenariusz B: kontrolowana zmiana (jasnosc LUB kontrast -- wybierz JEDNO,
# jedna sila, zdecydowana PRZED zobaczeniem accuracy w scenariuszu C)
# TODO:
# 1. zbuduj shifted_X (np. (X_control + SHIFT).clamp(X_control.min(), X_control.max()+1.0))
# 2. policz PSI(ref, shifted) -- czy wykrywa zmiane?
raise NotImplementedError("TODO: scenariusz B -- kontrolowana zmiana")

In [ ]:
# Scenariusz C: etykiety z opoznieniem -- masz y_control, wiec mozesz
# "symulowac" ich pozniejsza dostepnosc
# TODO:
# 1. policz accuracy modelu na X_control (referencja) i na shifted_X (z SAME y_control)
# 2. porownaj z wynikiem PSI ze scenariusza B -- drift i accuracy ida w te sama strone?
# 3. NIE zakladaj z gory odpowiedzi -- napisz, co SIE ZDARZYLO w Twoim eksperymencie
raise NotImplementedError("TODO: scenariusz C -- wplyw na jakosc")

## 8. Zadanie diagnostyczne: training-serving skew

`model_colleague.py` implementuje preprocessing **inaczej** niż Twój
`app/model.py`. Zobacz README sekcja 8.

In [ ]:
from model_colleague import (
    predict_colleague,  # noqa: F401 - uzywane w TODO ponizej
    preprocess_image_bytes_colleague,  # noqa: F401
)

print("model_colleague.py zaimportowany -- porownaj z Twoja implementacja.")

In [ ]:
# TODO:
# 1. na kilku (np. 10-20) prawdziwych obrazach EuroSAT, policz predict() (Twoje)
#    i predict_colleague() (kolegi) dla TYCH SAMYCH bytes
# 2. ile predykcji sie ROZNI (nie tylko logity)?
# 3. w komentarzu: dokladnie KTORY krok preprocessingu sie rozni, i dlaczego
#    to ma znaczenie (odwolaj sie do ResNet18_Weights.IMAGENET1K_V1.transforms())
raise NotImplementedError("TODO: diagnoza training-serving skew")

## 9. Raport: decyzja o wdrożeniu i utrzymaniu modelu — Twoje zadanie

**1. Wersja modelu, źródło danych, protokół ewaluacji offline.**

_(Twoja odpowiedź tutaj.)_

**2. Accuracy i najważniejsze ograniczenia.**

_(Twoja odpowiedź tutaj.)_

**3. Format wejścia/wyjścia API.**

_(Twoja odpowiedź tutaj.)_

**4. Latency p50/p95, throughput (Twoje zmierzone liczby).**

_(Twoja odpowiedź tutaj.)_

**5. Wyniki scenariuszy A/B/C.**

_(Twoja odpowiedź tutaj.)_

**6. Poziom pewności interpretacji driftu (jedna cecha, ograniczona próbka).**

_(Twoja odpowiedź tutaj.)_

**7. Znane ryzyka, rekomendowane następne działania.**

_(Twoja odpowiedź tutaj.)_